In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

import os
import numpy as np
import torch
from src.constants import BASEDIR, ALL_FEATURE_NAMES
import hydra
from datasets import interleave_datasets
from hydra import initialize_config_dir, compose
from src.data.collators import DocumentBatchCollator

os.chdir(BASEDIR)

DATASET_NAME = "foldseek_interleaved"
MAX_TOKENS = 8192
MASK_BELOW_PLDDT = 80

TODO: figure out how to change namespace config gets loaded to (@ - package)

critical for example data.

In [ ]:
with initialize_config_dir(os.path.join(BASEDIR, "configs")):
    cfg = compose(config_name=f"data/dataset/{DATASET_NAME}")
    tokenizer_cfg = compose(
        config_name="tokenizer/profam",
    )

In [ ]:
dataset_builder = hydra.utils.instantiate(cfg.data.dataset, _convert_="partial")
tokenizer = hydra.utils.instantiate(tokenizer_cfg.tokenizer)

In [ ]:
data = dataset_builder.load(data_dir="data/example_data", world_size=1)

In [ ]:
%%prun
b = next(iter(data))

In [ ]:
features = data.info.features

In [ ]:
features

In [ ]:
data = interleave_datasets([data, data])
data.features


In [ ]:
%%timeit -n 3 -r 1
b0 = next(iter(data))

In [ ]:
%%timeit
b1 = next(iter(data))

In [ ]:
collator = DocumentBatchCollator(tokenizer)

In [ ]:
%%timeit -n 3 -r 1
iterator = iter(data)
example = next(iterator)
batch = collator([example]*16)

In [ ]:
%%prun
iterator = iter(data)
example = next(iterator)
batch = collator([example]*16)

In [ ]:
iterator = iter(data)
example = next(iterator)

In [ ]:
example["coords"]

In [ ]:
example["aa_mask"]

In [ ]:
example["structure_mask"]

In [ ]:
np.array(example["interleaved_coords_mask"]).any((-1,-2)).sum()

In [ ]:
example["aa_mask"].sum()

In [ ]:
example["plddts"][2:50]

In [ ]:
example["structure_mask"].sum()

In [ ]:
example["high_plddt_mask"].sum()

In [ ]:
example.keys()

In [ ]:
batch = {k: v[None] for k, v in example.items() if torch.is_tensor(v)}

In [ ]:
example["input_ids"][:400]

In [ ]:
example["coords"][:400]

In [ ]:
example["plddt_mask"]

In [ ]:
example["structure_mask"]

In [ ]:
example["input_ids"]

TODO: add pymol visualisation.